In [55]:
# ============================================================
# 1. IMPORT LIBRARIES AND CONFIGURE THE TRANSFORMATION
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql import types as T

# Use UTC consistently for timestamps
spark.conf.set(
    "spark.sql.session.timeZone",
    "UTC"
)

# Project configuration
BRONZE_PATH = "Files/bronze/synthea"
SILVER_SCHEMA = "silver"
SOURCE_SYSTEM = "Synthea CSV November 2021"

SOURCE_FILE = "claims.csv"
SOURCE_TABLE = "claims"
TARGET_TABLE = f"{SILVER_SCHEMA}.claims"

# Create the Silver schema if needed
spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}"
)

print("Configuration completed.")
print(f"Bronze source: {BRONZE_PATH}")
print(f"Source file: {SOURCE_FILE}")
print(f"Target table: {TARGET_TABLE}")

StatementMeta(, bf4533da-0b67-44de-8670-44145449f8aa, 57, Finished, Available, Finished, False)

Configuration completed.
Bronze source: Files/bronze/synthea
Source file: claims.csv
Target table: silver.claims


In [56]:
# ============================================================
# 2. INGEST THE RAW BRONZE CLAIMS CSV FILE
# ============================================================

claim_file_path = (
    f"{BRONZE_PATH}/{SOURCE_FILE}"
)

claims_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv(
        claim_file_path
    )
)

raw_claim_count = (
    claims_raw.count()
)

print(
    f"Source table: {SOURCE_TABLE}"
)

print(
    f"Source file: {claim_file_path}"
)

print(
    f"Rows loaded: {raw_claim_count:,}"
)

print(
    f"Columns loaded: "
    f"{len(claims_raw.columns)}"
)

claims_raw.printSchema()

StatementMeta(, bf4533da-0b67-44de-8670-44145449f8aa, 58, Finished, Available, Finished, False)

Source table: claims
Source file: Files/bronze/synthea/claims.csv
Rows loaded: 117,889
Columns loaded: 31
root
 |-- Id: string (nullable = true)
 |-- PATIENTID: string (nullable = true)
 |-- PROVIDERID: string (nullable = true)
 |-- PRIMARYPATIENTINSURANCEID: string (nullable = true)
 |-- SECONDARYPATIENTINSURANCEID: string (nullable = true)
 |-- DEPARTMENTID: string (nullable = true)
 |-- PATIENTDEPARTMENTID: string (nullable = true)
 |-- DIAGNOSIS1: string (nullable = true)
 |-- DIAGNOSIS2: string (nullable = true)
 |-- DIAGNOSIS3: string (nullable = true)
 |-- DIAGNOSIS4: string (nullable = true)
 |-- DIAGNOSIS5: string (nullable = true)
 |-- DIAGNOSIS6: string (nullable = true)
 |-- DIAGNOSIS7: string (nullable = true)
 |-- DIAGNOSIS8: string (nullable = true)
 |-- REFERRINGPROVIDERID: string (nullable = true)
 |-- APPOINTMENTID: string (nullable = true)
 |-- CURRENTILLNESSDATE: string (nullable = true)
 |-- SERVICEDATE: string (nullable = true)
 |-- SUPERVISINGPROVIDERID: string (

In [57]:
# ============================================================
# 3. INSPECT RAW CLAIM RECORDS
# ============================================================

display(
    claims_raw.limit(10)
)

print(
    claims_raw.columns
)

StatementMeta(, bf4533da-0b67-44de-8670-44145449f8aa, 59, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b9d68cc3-a072-4c5e-87e3-d520af55eba4)

['Id', 'PATIENTID', 'PROVIDERID', 'PRIMARYPATIENTINSURANCEID', 'SECONDARYPATIENTINSURANCEID', 'DEPARTMENTID', 'PATIENTDEPARTMENTID', 'DIAGNOSIS1', 'DIAGNOSIS2', 'DIAGNOSIS3', 'DIAGNOSIS4', 'DIAGNOSIS5', 'DIAGNOSIS6', 'DIAGNOSIS7', 'DIAGNOSIS8', 'REFERRINGPROVIDERID', 'APPOINTMENTID', 'CURRENTILLNESSDATE', 'SERVICEDATE', 'SUPERVISINGPROVIDERID', 'STATUS1', 'STATUS2', 'STATUSP', 'OUTSTANDING1', 'OUTSTANDING2', 'OUTSTANDINGP', 'LASTBILLEDDATE1', 'LASTBILLEDDATE2', 'LASTBILLEDDATEP', 'HEALTHCARECLAIMTYPEID1', 'HEALTHCARECLAIMTYPEID2']


In [58]:
# ============================================================
# 4. STANDARDIZE CLAIM COLUMN NAMES
# ============================================================

claim_column_mapping = {
    "Id": "claim_id",
    "ID": "claim_id",

    "PATIENTID": "patient_id",
    "PROVIDERID": "provider_id",

    "PRIMARYPATIENTINSURANCEID":
        "primary_payer_id",

    "SECONDARYPATIENTINSURANCEID":
        "secondary_payer_id",

    "DEPARTMENTID": "department_id",
    "PATIENTDEPARTMENTID":
        "patient_department_id",

    "DIAGNOSIS1": "diagnosis_code_1",
    "DIAGNOSIS2": "diagnosis_code_2",
    "DIAGNOSIS3": "diagnosis_code_3",
    "DIAGNOSIS4": "diagnosis_code_4",
    "DIAGNOSIS5": "diagnosis_code_5",
    "DIAGNOSIS6": "diagnosis_code_6",
    "DIAGNOSIS7": "diagnosis_code_7",
    "DIAGNOSIS8": "diagnosis_code_8",

    "REFERRINGPROVIDERID":
        "referring_provider_id",

    # This is an encounter relationship,
    # not the claim primary key
    "APPOINTMENTID": "encounter_id",

    "CURRENTILLNESSDATE":
        "current_illness_timestamp",

    "SERVICEDATE":
        "service_timestamp",

    "SUPERVISINGPROVIDERID":
        "supervising_provider_id",

    "STATUS1": "primary_claim_status",
    "STATUS2": "secondary_claim_status",
    "STATUSP": "patient_claim_status",

    "OUTSTANDING1":
        "primary_outstanding_amount",

    "OUTSTANDING2":
        "secondary_outstanding_amount",

    "OUTSTANDINGP":
        "patient_outstanding_amount",

    "LASTBILLEDDATE1":
        "primary_last_billed_timestamp",

    "LASTBILLEDDATE2":
        "secondary_last_billed_timestamp",

    "LASTBILLEDDATEP":
        "patient_last_billed_timestamp",

    "HEALTHCARECLAIMTYPEID1":
        "primary_claim_type_id",

    "HEALTHCARECLAIMTYPEID2":
        "secondary_claim_type_id"
}


claims_standardized = claims_raw


for old_name in claims_raw.columns:

    if old_name in claim_column_mapping:

        new_name = claim_column_mapping[
            old_name
        ]

    else:

        new_name = (
            old_name
            .strip()
            .lower()
        )

    claims_standardized = (
        claims_standardized
        .withColumnRenamed(
            old_name,
            new_name
        )
    )


# Stop if standardization creates
# duplicate column names
if (
    len(claims_standardized.columns)
    !=
    len(set(claims_standardized.columns))
):

    raise ValueError(
        "Duplicate column names were created "
        "while standardizing claims."
    )


print(
    f"Claims: "
    f"{len(claims_raw.columns)} source columns -> "
    f"{len(claims_standardized.columns)} "
    f"standardized columns"
)

print(
    claims_standardized.columns
)

StatementMeta(, bf4533da-0b67-44de-8670-44145449f8aa, 60, Finished, Available, Finished, False)

Claims: 31 source columns -> 31 standardized columns
['claim_id', 'patient_id', 'provider_id', 'primary_payer_id', 'secondary_payer_id', 'department_id', 'patient_department_id', 'diagnosis_code_1', 'diagnosis_code_2', 'diagnosis_code_3', 'diagnosis_code_4', 'diagnosis_code_5', 'diagnosis_code_6', 'diagnosis_code_7', 'diagnosis_code_8', 'referring_provider_id', 'encounter_id', 'current_illness_timestamp', 'service_timestamp', 'supervising_provider_id', 'primary_claim_status', 'secondary_claim_status', 'patient_claim_status', 'primary_outstanding_amount', 'secondary_outstanding_amount', 'patient_outstanding_amount', 'primary_last_billed_timestamp', 'secondary_last_billed_timestamp', 'patient_last_billed_timestamp', 'primary_claim_type_id', 'secondary_claim_type_id']


In [59]:
# ============================================================
# 5. CLEAN CLAIM STRING VALUES
# ============================================================


def clean_string_values(df):

    cleaned_df = df

    for column_name in df.columns:

        cleaned_df = (
            cleaned_df
            .withColumn(
                column_name,
                F.when(
                    F.trim(
                        F.col(column_name)
                    ) == "",
                    F.lit(None)
                )
                .otherwise(
                    F.trim(
                        F.col(column_name)
                    )
                )
            )
        )

    return cleaned_df


# Count records before cleaning
claim_count_before_cleaning = (
    claims_standardized.count()
)


# Trim text and convert blank values to null
claims_cleaned = clean_string_values(
    claims_standardized
)


# Count records after cleaning
claim_count_after_cleaning = (
    claims_cleaned.count()
)


print(
    f"Claims: "
    f"{claim_count_before_cleaning:,} rows "
    f"before cleaning, "
    f"{claim_count_after_cleaning:,} rows "
    f"after cleaning"
)


# Stop if the cleaning operation changes
# the number of claim records
if (
    claim_count_before_cleaning
    !=
    claim_count_after_cleaning
):

    raise ValueError(
        "Claim row count changed during "
        "string-value cleaning."
    )


print(
    "Claim string cleaning and "
    "row-count reconciliation passed."
)


StatementMeta(, bf4533da-0b67-44de-8670-44145449f8aa, 61, Finished, Available, Finished, False)

Claims: 117,889 rows before cleaning, 117,889 rows after cleaning
Claim string cleaning and row-count reconciliation passed.


In [60]:
# ============================================================
# 6. TRANSFORM CLAIM DATA TYPES
# ============================================================

claims_silver = (
    claims_cleaned
        # Preserve original payer values for auditing
    .withColumn(
        "source_primary_payer_id",
        F.col("primary_payer_id")
    )
    .withColumn(
        "source_secondary_payer_id",
        F.col("secondary_payer_id")
    )

    # Normalize the source value "0" to null because
    # it does not represent a payer record
    .withColumn(
        "primary_payer_id",
        F.when(
            F.col("primary_payer_id") == "0",
            F.lit(None).cast("string")
        ).otherwise(
            F.col("primary_payer_id")
        )
    )
    .withColumn(
        "secondary_payer_id",
        F.when(
            F.col("secondary_payer_id") == "0",
            F.lit(None).cast("string")
        ).otherwise(
            F.col("secondary_payer_id")
        )
    )

    # Classify claims based on payer availability
    .withColumn(
        "claim_coverage_category",
        F.when(
            F.col("primary_payer_id").isNull(),
            F.lit("self-pay or uninsured")
        ).otherwise(
            F.lit("insured")
        )
    )
    # Convert illness and service values
    # from strings to UTC timestamps
    .withColumn(
        "current_illness_timestamp",
        F.to_timestamp(
            F.col(
                "current_illness_timestamp"
            ),
            "yyyy-MM-dd'T'HH:mm:ssX"
        )
    )
    .withColumn(
        "service_timestamp",
        F.to_timestamp(
            F.col(
                "service_timestamp"
            ),
            "yyyy-MM-dd'T'HH:mm:ssX"
        )
    )

    # Convert last-billed values to timestamps
    .withColumn(
        "primary_last_billed_timestamp",
        F.to_timestamp(
            F.col(
                "primary_last_billed_timestamp"
            ),
            "yyyy-MM-dd'T'HH:mm:ssX"
        )
    )
    .withColumn(
        "secondary_last_billed_timestamp",
        F.to_timestamp(
            F.col(
                "secondary_last_billed_timestamp"
            ),
            "yyyy-MM-dd'T'HH:mm:ssX"
        )
    )
    .withColumn(
        "patient_last_billed_timestamp",
        F.to_timestamp(
            F.col(
                "patient_last_billed_timestamp"
            ),
            "yyyy-MM-dd'T'HH:mm:ssX"
        )
    )

    # Convert outstanding balances to
    # fixed-precision decimal values
    .withColumn(
        "primary_outstanding_amount",
        F.col(
            "primary_outstanding_amount"
        ).cast(
            T.DecimalType(18, 2)
        )
    )
    .withColumn(
        "secondary_outstanding_amount",
        F.col(
            "secondary_outstanding_amount"
        ).cast(
            T.DecimalType(18, 2)
        )
    )
    .withColumn(
        "patient_outstanding_amount",
        F.col(
            "patient_outstanding_amount"
        ).cast(
            T.DecimalType(18, 2)
        )
    )

    # Standardize claim-status values
    .withColumn(
        "primary_claim_status",
        F.lower(
            F.col("primary_claim_status")
        )
    )
    .withColumn(
        "secondary_claim_status",
        F.lower(
            F.col("secondary_claim_status")
        )
    )
    .withColumn(
        "patient_claim_status",
        F.lower(
            F.col("patient_claim_status")
        )
    )

    # Create the service date for reporting
    .withColumn(
        "service_date",
        F.to_date(
            F.col("service_timestamp")
        )
    )

    # Calculate total outstanding balance
    # across primary, secondary and patient portions
    .withColumn(
        "total_outstanding_amount",
        (
            F.coalesce(
                F.col(
                    "primary_outstanding_amount"
                ),
                F.lit(0).cast(
                    T.DecimalType(18, 2)
                )
            )
            +
            F.coalesce(
                F.col(
                    "secondary_outstanding_amount"
                ),
                F.lit(0).cast(
                    T.DecimalType(18, 2)
                )
            )
            +
            F.coalesce(
                F.col(
                    "patient_outstanding_amount"
                ),
                F.lit(0).cast(
                    T.DecimalType(18, 2)
                )
            )
        ).cast(
            T.DecimalType(18, 2)
        )
    )
)


claims_silver.printSchema()

StatementMeta(, bf4533da-0b67-44de-8670-44145449f8aa, 62, Finished, Available, Finished, False)

root
 |-- claim_id: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- provider_id: string (nullable = true)
 |-- primary_payer_id: string (nullable = true)
 |-- secondary_payer_id: string (nullable = true)
 |-- department_id: string (nullable = true)
 |-- patient_department_id: string (nullable = true)
 |-- diagnosis_code_1: string (nullable = true)
 |-- diagnosis_code_2: string (nullable = true)
 |-- diagnosis_code_3: string (nullable = true)
 |-- diagnosis_code_4: string (nullable = true)
 |-- diagnosis_code_5: string (nullable = true)
 |-- diagnosis_code_6: string (nullable = true)
 |-- diagnosis_code_7: string (nullable = true)
 |-- diagnosis_code_8: string (nullable = true)
 |-- referring_provider_id: string (nullable = true)
 |-- encounter_id: string (nullable = true)
 |-- current_illness_timestamp: timestamp (nullable = true)
 |-- service_timestamp: timestamp (nullable = true)
 |-- supervising_provider_id: string (nullable = true)
 |-- primary_claim_status:

In [62]:
# ============================================================
# 7. VALIDATE CLAIM DATA QUALITY
# ============================================================

claim_validation = (
    claims_silver
    .agg(
        # Row-count and identifier validations
        F.count("*").alias(
            "row_count"
        ),

        F.countDistinct(
            "claim_id"
        ).alias(
            "distinct_claim_ids"
        ),

        # encounter_id is a foreign key and is expected
        # to occur across multiple claim records
        F.countDistinct(
            "encounter_id"
        ).alias(
            "distinct_encounter_ids"
        ),

        F.sum(
            F.when(
                F.col("claim_id").isNull(),
                1
            ).otherwise(0)
        ).alias(
            "null_claim_ids"
        ),

        F.sum(
            F.when(
                F.col("patient_id").isNull(),
                1
            ).otherwise(0)
        ).alias(
            "null_patient_ids"
        ),

        F.sum(
            F.when(
                F.col("encounter_id").isNull(),
                1
            ).otherwise(0)
        ).alias(
            "null_encounter_ids"
        ),

        # A null primary payer represents a source value
        # of "0", meaning self-pay or uninsured
        F.sum(
            F.when(
                F.col("primary_payer_id").isNull(),
                1
            ).otherwise(0)
        ).alias(
            "self_pay_or_uninsured_claims"
        ),

        # Timestamp validations
        F.sum(
            F.when(
                F.col("current_illness_timestamp").isNull(),
                1
            ).otherwise(0)
        ).alias(
            "null_current_illness_timestamps"
        ),

        F.sum(
            F.when(
                F.col("service_timestamp").isNull(),
                1
            ).otherwise(0)
        ).alias(
            "null_service_timestamps"
        ),

        F.sum(
            F.when(
                F.col("current_illness_timestamp")
                >
                F.col("service_timestamp"),
                1
            ).otherwise(0)
        ).alias(
            "illness_after_service"
        ),

        # Financial validations
        F.sum(
            F.when(
                F.col("primary_outstanding_amount") < 0,
                1
            ).otherwise(0)
        ).alias(
            "negative_primary_outstanding"
        ),

        F.sum(
            F.when(
                F.col("secondary_outstanding_amount") < 0,
                1
            ).otherwise(0)
        ).alias(
            "negative_secondary_outstanding"
        ),

        F.sum(
            F.when(
                F.col("patient_outstanding_amount") < 0,
                1
            ).otherwise(0)
        ).alias(
            "negative_patient_outstanding"
        ),

        F.sum(
            F.when(
                F.col("total_outstanding_amount") < 0,
                1
            ).otherwise(0)
        ).alias(
            "negative_total_outstanding"
        )
    )
)

display(claim_validation)

StatementMeta(, bf4533da-0b67-44de-8670-44145449f8aa, 64, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b75fbaeb-60c1-4d13-af77-6e5bc8f11cd0)

In [63]:
# ============================================================
# 7.1 ENFORCE CRITICAL CLAIM VALIDATIONS
# ============================================================

claim_validation_result = claim_validation.first()

# Confirm expected row count
assert claim_validation_result["row_count"] == 117889, (
    f"Expected 117,889 claims but found "
    f"{claim_validation_result['row_count']:,}."
)

# Confirm claim_id is unique
assert claim_validation_result["distinct_claim_ids"] == 117889, (
    "Claim IDs are not unique."
)

# Confirm required identifiers are populated
assert claim_validation_result["null_claim_ids"] == 0, (
    "Null claim IDs were detected."
)

assert claim_validation_result["null_patient_ids"] == 0, (
    "Null patient IDs were detected."
)

assert claim_validation_result["null_encounter_ids"] == 0, (
    "Null encounter IDs were detected."
)

# Confirm timestamps were parsed successfully
assert claim_validation_result[
    "null_current_illness_timestamps"
] == 0, (
    "Null or unparseable illness timestamps were detected."
)

assert claim_validation_result[
    "null_service_timestamps"
] == 0, (
    "Null or unparseable service timestamps were detected."
)

# Confirm timestamp sequence is valid
assert claim_validation_result["illness_after_service"] == 0, (
    "Claims were found where illness time occurs after service time."
)

# Confirm outstanding amounts are nonnegative
assert claim_validation_result[
    "negative_primary_outstanding"
] == 0, (
    "Negative primary outstanding amounts were detected."
)

assert claim_validation_result[
    "negative_secondary_outstanding"
] == 0, (
    "Negative secondary outstanding amounts were detected."
)

assert claim_validation_result[
    "negative_patient_outstanding"
] == 0, (
    "Negative patient outstanding amounts were detected."
)

assert claim_validation_result[
    "negative_total_outstanding"
] == 0, (
    "Negative total outstanding amounts were detected."
)

# Report the documented self-pay/uninsured population
self_pay_claim_count = claim_validation_result[
    "self_pay_or_uninsured_claims"
]

insured_claim_count = (
    claim_validation_result["row_count"]
    - self_pay_claim_count
)

print("Critical claim data-quality validations passed.")
print(f"Insured claims: {insured_claim_count:,}")
print(
    f"Self-pay or uninsured claims: "
    f"{self_pay_claim_count:,}"
)
print(
    "The source payer value '0' was normalized to null "
    "and classified as self-pay or uninsured."
)
print(
    "Encounter IDs are intentionally nonunique because "
    "multiple claims can relate to the same encounter."
)

StatementMeta(, bf4533da-0b67-44de-8670-44145449f8aa, 65, Finished, Available, Finished, False)

Critical claim data-quality validations passed.
Insured claims: 91,922
Self-pay or uninsured claims: 25,967
The source payer value '0' was normalized to null and classified as self-pay or uninsured.
Encounter IDs are intentionally nonunique because multiple claims can relate to the same encounter.


In [64]:
# ============================================================
# 7.2 VALIDATE CLAIM-TO-PATIENT RELATIONSHIPS
# ============================================================

# Retrieve valid patient identifiers from the Silver patient table
valid_patients = (
    spark.table(
        f"{SILVER_SCHEMA}.patients"
    )
    .select(
        F.col("id").alias("valid_patient_id")
    )
    .distinct()
)

# Find claims whose patient_id does not exist in silver.patients
invalid_claim_patients = (
    claims_silver
    .select(
        "claim_id",
        "patient_id"
    )
    .join(
        valid_patients,
        F.col("patient_id")
        ==
        F.col("valid_patient_id"),
        "left_anti"
    )
)

invalid_claim_patient_count = (
    invalid_claim_patients.count()
)

print(
    f"Claims with an invalid patient ID: "
    f"{invalid_claim_patient_count:,}"
)

# Stop the notebook if invalid relationships exist
if invalid_claim_patient_count > 0:
    display(
        invalid_claim_patients.limit(20)
    )

    raise ValueError(
        "Claim-to-patient foreign-key validation failed."
    )

print(
    "Claim-to-patient foreign-key validation passed."
)

StatementMeta(, bf4533da-0b67-44de-8670-44145449f8aa, 66, Finished, Available, Finished, False)

Claims with an invalid patient ID: 0
Claim-to-patient foreign-key validation passed.


In [65]:
# ============================================================
# 7.3 VALIDATE CLAIM-TO-ENCOUNTER RELATIONSHIPS
# ============================================================

# Retrieve valid encounter identifiers
valid_encounters = (
    spark.table(
        f"{SILVER_SCHEMA}.encounters"
    )
    .select(
        F.col("encounter_id").alias(
            "valid_encounter_id"
        )
    )
    .distinct()
)

# Find claim records without a matching encounter
invalid_claim_encounters = (
    claims_silver
    .select(
        "claim_id",
        "encounter_id"
    )
    .join(
        valid_encounters,
        F.col("encounter_id")
        ==
        F.col("valid_encounter_id"),
        "left_anti"
    )
)

invalid_claim_encounter_count = (
    invalid_claim_encounters.count()
)

print(
    f"Claims with an invalid encounter ID: "
    f"{invalid_claim_encounter_count:,}"
)

# Stop if invalid encounter relationships exist
if invalid_claim_encounter_count > 0:
    display(
        invalid_claim_encounters.limit(20)
    )

    raise ValueError(
        "Claim-to-encounter foreign-key validation failed."
    )

print(
    "Claim-to-encounter foreign-key validation passed."
)

StatementMeta(, bf4533da-0b67-44de-8670-44145449f8aa, 67, Finished, Available, Finished, False)

Claims with an invalid encounter ID: 0
Claim-to-encounter foreign-key validation passed.


In [66]:
# ============================================================
# 7.4 VALIDATE CLAIM PRIMARY-PAYER RELATIONSHIPS
# ============================================================

# Retrieve valid payer identifiers
valid_payers = (
    spark.table(
        f"{SILVER_SCHEMA}.payers"
    )
    .select(
        F.col("payer_id").alias(
            "valid_payer_id"
        )
    )
    .distinct()
)

# Separate claims that contain a primary payer
insured_claims_for_validation = (
    claims_silver
    .filter(
        F.col("primary_payer_id").isNotNull()
    )
)

# Find non-null primary payer IDs that do not exist
# in the Silver payer table
invalid_primary_claim_payers = (
    insured_claims_for_validation
    .select(
        "claim_id",
        "primary_payer_id"
    )
    .join(
        valid_payers,
        F.col("primary_payer_id")
        ==
        F.col("valid_payer_id"),
        "left_anti"
    )
)

insured_claim_count = (
    insured_claims_for_validation.count()
)

self_pay_claim_count = (
    claims_silver
    .filter(
        F.col("primary_payer_id").isNull()
    )
    .count()
)

invalid_primary_payer_count = (
    invalid_primary_claim_payers.count()
)

print(
    f"Claims with a primary payer: "
    f"{insured_claim_count:,}"
)

print(
    f"Self-pay or uninsured claims: "
    f"{self_pay_claim_count:,}"
)

print(
    f"Claims with an invalid non-null primary payer ID: "
    f"{invalid_primary_payer_count:,}"
)

if invalid_primary_payer_count > 0:
    display(
        invalid_primary_claim_payers.limit(20)
    )

    raise ValueError(
        "Claim primary-payer foreign-key validation failed."
    )

print(
    "Claim primary-payer foreign-key validation passed."
)

StatementMeta(, bf4533da-0b67-44de-8670-44145449f8aa, 68, Finished, Available, Finished, False)

Claims with a primary payer: 91,922
Self-pay or uninsured claims: 25,967
Claims with an invalid non-null primary payer ID: 0
Claim primary-payer foreign-key validation passed.


In [67]:
# ============================================================
# 7.5 VALIDATE CLAIM SECONDARY-PAYER RELATIONSHIPS
# ============================================================

# Select claims containing a secondary payer
claims_with_secondary_payer = (
    claims_silver
    .filter(
        F.col("secondary_payer_id").isNotNull()
    )
)

# Find non-null secondary payer IDs that do not exist
# in the Silver payer table
invalid_secondary_claim_payers = (
    claims_with_secondary_payer
    .select(
        "claim_id",
        "secondary_payer_id"
    )
    .join(
        valid_payers,
        F.col("secondary_payer_id")
        ==
        F.col("valid_payer_id"),
        "left_anti"
    )
)

secondary_payer_claim_count = (
    claims_with_secondary_payer.count()
)

claims_without_secondary_payer_count = (
    claims_silver
    .filter(
        F.col("secondary_payer_id").isNull()
    )
    .count()
)

invalid_secondary_payer_count = (
    invalid_secondary_claim_payers.count()
)

print(
    f"Claims with a secondary payer: "
    f"{secondary_payer_claim_count:,}"
)

print(
    f"Claims without a secondary payer: "
    f"{claims_without_secondary_payer_count:,}"
)

print(
    f"Claims with an invalid non-null secondary payer ID: "
    f"{invalid_secondary_payer_count:,}"
)

if invalid_secondary_payer_count > 0:
    display(
        invalid_secondary_claim_payers.limit(20)
    )

    raise ValueError(
        "Claim secondary-payer foreign-key validation failed."
    )

print(
    "Claim secondary-payer foreign-key validation passed."
)

StatementMeta(, bf4533da-0b67-44de-8670-44145449f8aa, 69, Finished, Available, Finished, False)

Claims with a secondary payer: 14,170
Claims without a secondary payer: 103,719
Claims with an invalid non-null secondary payer ID: 0
Claim secondary-payer foreign-key validation passed.


In [68]:
# ============================================================
# 8. VALIDATE DUPLICATES AND CLAIM PRIMARY KEY
# ============================================================

# Check for exact duplicate source records
exact_duplicate_claim_rows = (
    claims_cleaned
    .groupBy(
        *claims_cleaned.columns
    )
    .count()
    .filter(
        F.col("count") > 1
    )
)

exact_duplicate_group_count = (
    exact_duplicate_claim_rows.count()
)

exact_duplicate_extra_row_count = (
    exact_duplicate_claim_rows
    .select(
        F.coalesce(
            F.sum(
                F.col("count") - 1
            ),
            F.lit(0)
        ).alias("duplicate_extra_rows")
    )
    .first()["duplicate_extra_rows"]
)

# Check for duplicate claim IDs
duplicate_claim_ids = (
    claims_silver
    .groupBy("claim_id")
    .count()
    .filter(
        F.col("count") > 1
    )
)

duplicate_claim_id_count = (
    duplicate_claim_ids.count()
)

# Identify encounter IDs connected to multiple claims
encounters_with_multiple_claims = (
    claims_silver
    .groupBy("encounter_id")
    .count()
    .filter(
        F.col("count") > 1
    )
)

multi_claim_encounter_count = (
    encounters_with_multiple_claims.count()
)

maximum_claims_per_encounter = (
    claims_silver
    .groupBy("encounter_id")
    .count()
    .agg(
        F.max("count").alias(
            "maximum_claims_per_encounter"
        )
    )
    .first()["maximum_claims_per_encounter"]
)

print(
    f"Exact duplicate claim groups: "
    f"{exact_duplicate_group_count:,}"
)

print(
    f"Extra exact duplicate claim rows: "
    f"{exact_duplicate_extra_row_count:,}"
)

print(
    f"Duplicated claim IDs: "
    f"{duplicate_claim_id_count:,}"
)

print(
    f"Encounter IDs linked to multiple claims: "
    f"{multi_claim_encounter_count:,}"
)

print(
    f"Maximum claims for one encounter: "
    f"{maximum_claims_per_encounter:,}"
)

# Enforce the duplicate and primary-key rules
if exact_duplicate_group_count > 0:
    display(
        exact_duplicate_claim_rows.limit(20)
    )

    raise ValueError(
        "Exact duplicate claim records were detected."
    )

if duplicate_claim_id_count > 0:
    display(
        duplicate_claim_ids.limit(20)
    )

    raise ValueError(
        "Duplicate claim IDs were detected."
    )

if multi_claim_encounter_count == 0:
    raise ValueError(
        "Expected encounter_id to be nonunique, "
        "but no repeated encounter IDs were found."
    )

print(
    "No exact duplicate claim rows were detected."
)

print(
    "claim_id passed the primary-key uniqueness validation."
)

print(
    "encounter_id is nonunique and will remain a foreign key; "
    "it must not be used as the claims primary key."
)

StatementMeta(, bf4533da-0b67-44de-8670-44145449f8aa, 70, Finished, Available, Finished, False)

Exact duplicate claim groups: 0
Extra exact duplicate claim rows: 0
Duplicated claim IDs: 0
Encounter IDs linked to multiple claims: 26,871
Maximum claims for one encounter: 14
No exact duplicate claim rows were detected.
claim_id passed the primary-key uniqueness validation.
encounter_id is nonunique and will remain a foreign key; it must not be used as the claims primary key.


In [69]:
# ============================================================
# 9. ADD RECORD HASH AND AUDIT METADATA
# ============================================================

# Use all current business columns when creating the hash
claim_hash_columns = claims_silver.columns

# Convert every value to text and represent nulls consistently
claim_hash_inputs = [
    F.coalesce(
        F.col(column_name).cast("string"),
        F.lit("<NULL>")
    )
    for column_name in claim_hash_columns
]

# Generate a deterministic SHA-256 record hash
claims_silver_final = (
    claims_silver
    .withColumn(
        "record_hash",
        F.sha2(
            F.concat_ws(
                "||",
                *claim_hash_inputs
            ),
            256
        )
    )

    # Add data-lineage and processing metadata
    .withColumn(
        "source_system",
        F.lit(SOURCE_SYSTEM)
    )
    .withColumn(
        "source_file",
        F.lit(SOURCE_FILE)
    )
    .withColumn(
        "silver_processed_timestamp",
        F.current_timestamp()
    )
)

print(
    f"Hash input columns: "
    f"{len(claim_hash_columns)}"
)

print(
    f"Final Silver columns: "
    f"{len(claims_silver_final.columns)}"
)

claims_silver_final.select(
    "claim_id",
    "encounter_id",
    "claim_coverage_category",
    "record_hash",
    "source_system",
    "source_file",
    "silver_processed_timestamp"
).show(
    5,
    truncate=False
)

StatementMeta(, bf4533da-0b67-44de-8670-44145449f8aa, 71, Finished, Available, Finished, False)

Hash input columns: 36
Final Silver columns: 40
+------------------------------------+------------------------------------+-----------------------+----------------------------------------------------------------+-------------------------+-----------+--------------------------+
|claim_id                            |encounter_id                        |claim_coverage_category|record_hash                                                     |source_system            |source_file|silver_processed_timestamp|
+------------------------------------+------------------------------------+-----------------------+----------------------------------------------------------------+-------------------------+-----------+--------------------------+
|e413105d-6f23-34ef-724a-b42adab9df22|748f8357-6cc7-551d-f31a-32fa2cf84126|insured                |ca290bd17f35cb0c3751c34ebf176057a4ba55f8fde30123f174fd99785e98df|Synthea CSV November 2021|claims.csv |2026-10-21 06:22:27.547988|
|8e430d76-6628-c3ac-8950-2acabeb

In [70]:
# ============================================================
# 9.1 VALIDATE RECORD HASHES AND AUDIT METADATA
# ============================================================

claim_hash_validation = (
    claims_silver_final
    .agg(
        F.count("*").alias(
            "final_row_count"
        ),

        F.sum(
            F.when(
                F.col("record_hash").isNull(),
                1
            ).otherwise(0)
        ).alias(
            "null_record_hashes"
        ),

        F.sum(
            F.when(
                F.length("record_hash") != 64,
                1
            ).otherwise(0)
        ).alias(
            "invalid_hash_lengths"
        ),

        F.sum(
            F.when(
                F.col("source_system").isNull(),
                1
            ).otherwise(0)
        ).alias(
            "null_source_systems"
        ),

        F.sum(
            F.when(
                F.col("source_file").isNull(),
                1
            ).otherwise(0)
        ).alias(
            "null_source_files"
        ),

        F.sum(
            F.when(
                F.col(
                    "silver_processed_timestamp"
                ).isNull(),
                1
            ).otherwise(0)
        ).alias(
            "null_processed_timestamps"
        )
    )
)

display(claim_hash_validation)

claim_hash_result = claim_hash_validation.first()

assert claim_hash_result["final_row_count"] == 117889, (
    "The final claim row count does not match the source."
)

assert claim_hash_result["null_record_hashes"] == 0, (
    "Null claim record hashes were detected."
)

assert claim_hash_result["invalid_hash_lengths"] == 0, (
    "Invalid SHA-256 hash lengths were detected."
)

assert claim_hash_result["null_source_systems"] == 0, (
    "Null source-system metadata was detected."
)

assert claim_hash_result["null_source_files"] == 0, (
    "Null source-file metadata was detected."
)

assert claim_hash_result["null_processed_timestamps"] == 0, (
    "Null Silver processing timestamps were detected."
)

print(
    "Claim record-hash and audit-metadata "
    "validations passed."
)

print(
    f"Final claims ready for writing: "
    f"{claim_hash_result['final_row_count']:,}"
)

StatementMeta(, bf4533da-0b67-44de-8670-44145449f8aa, 72, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f189b99e-eaed-44fc-a1cb-2f6c2e7adc77)

Claim record-hash and audit-metadata validations passed.
Final claims ready for writing: 117,889


In [71]:
# ============================================================
# 10. WRITE CLAIMS TO THE SILVER DELTA TABLE
# ============================================================

(
    claims_silver_final
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(
        TARGET_TABLE
    )
)

print(
    f"Silver claims table written successfully: "
    f"{TARGET_TABLE}"
)

StatementMeta(, bf4533da-0b67-44de-8670-44145449f8aa, 73, Finished, Available, Finished, False)

Silver claims table written successfully: silver.claims


In [72]:
# ============================================================
# 11. VERIFY THE SAVED SILVER CLAIMS TABLE
# ============================================================

saved_claims = (
    spark.table(
        TARGET_TABLE
    )
)

saved_claim_validation = (
    saved_claims
    .agg(
        F.count("*").alias(
            "saved_row_count"
        ),

        F.countDistinct(
            "claim_id"
        ).alias(
            "saved_distinct_claim_ids"
        ),

        F.sum(
            F.when(
                F.col("record_hash").isNull(),
                1
            ).otherwise(0)
        ).alias(
            "saved_null_hashes"
        ),

        F.sum(
            F.when(
                F.col("primary_payer_id").isNull(),
                1
            ).otherwise(0)
        ).alias(
            "saved_self_pay_or_uninsured_claims"
        )
    )
)

display(saved_claim_validation)

saved_claim_result = saved_claim_validation.first()

assert saved_claim_result["saved_row_count"] == 117889, (
    "Saved Silver claims row count is incorrect."
)

assert saved_claim_result[
    "saved_distinct_claim_ids"
] == 117889, (
    "Saved Silver claim IDs are not unique."
)

assert saved_claim_result["saved_null_hashes"] == 0, (
    "Null record hashes were found in the saved table."
)

assert saved_claim_result[
    "saved_self_pay_or_uninsured_claims"
] == 25967, (
    "The saved self-pay/uninsured claim count is incorrect."
)

print("Silver claims table verification passed.")
print(
    f"Saved rows: "
    f"{saved_claim_result['saved_row_count']:,}"
)
print(
    f"Unique claim IDs: "
    f"{saved_claim_result['saved_distinct_claim_ids']:,}"
)
print(
    f"Self-pay or uninsured claims: "
    f"{saved_claim_result['saved_self_pay_or_uninsured_claims']:,}"
)

display(
    saved_claims.select(
        "claim_id",
        "patient_id",
        "encounter_id",
        "primary_payer_id",
        "claim_coverage_category",
        "service_timestamp",
        "total_outstanding_amount",
        "record_hash",
        "source_file",
        "silver_processed_timestamp"
    ).limit(10)
)

StatementMeta(, bf4533da-0b67-44de-8670-44145449f8aa, 74, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ad61207d-e6e4-4d37-b3e5-66d7c8c1c81f)

Silver claims table verification passed.
Saved rows: 117,889
Unique claim IDs: 117,889
Self-pay or uninsured claims: 25,967


SynapseWidget(Synapse.DataFrame, 900e0ab4-a4db-4f1b-8a14-32f3a55ae16e)